# RadSim: End-to-End Gamma-Ray Detector Simulation

This notebook demonstrates the complete end-to-end workflow for simulating gamma-ray detector responses using the open-source RadSim framework. We will cover:

1. **Source Definition**: Define radiation sources and calculate emissions using nuclear data
2. **Flux Transport**: Model radiation transport through materials using GEANT4
3. **Detector Response**: Simulate detector response using the ML-accelerated flow matching technique

This demonstration shows RadSim's modular design and how components can be combined for comprehensive radiation simulations.

## Setup and Initialization

First, we'll import necessary libraries and initialize the JVM for JPype.

In [ ]:
import startJVM
import jpype
import jpype.imports
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
import pandas as pd

# Configure plot style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

Import the necessary Java classes for our simulation workflow:

In [ ]:
# Source and emission calculation
from gov.llnl.rtk.physics import Nuclides
from gov.llnl.rtk.physics import SourceImpl
from gov.llnl.rtk.physics import ActivityUnit
from gov.llnl.rtk.physics import EmissionCalculator
from gov.llnl.rtk.physics import DecayCalculator
from gov.llnl.ensdf.decay import BNLDecayLibrary
from java.nio.file import Paths

# Flux representation
from gov.llnl.rtk.flux import FluxTrapezoid, FluxGroupTrapezoid
from java.util import ArrayList

# Material cross-sections
from gov.nist.physics.xcom import XCOMPhotonCrossSectionLibrary
from gov.llnl.rtk.physics import Elements

# Detector response
from gov.llnl.rtk.physics import Quantity, DBKNDistribution, ElectronShell

# Load GEANT4 integration
SourceGenerator = jpype.JClass('gov.llnl.rtk.geant4.SourceGenerator')

# Load response module components
DepositionCalculator = jpype.JClass('gov.llnl.rtk.response.deposition.DepositionCalculator')
CylinderChordGeometry = jpype.JClass('gov.llnl.rtk.response.geometry.CylinderChordGeometry')
CuboidChordGeometry = jpype.JClass('gov.llnl.rtk.response.geometry.CuboidChordGeometry')

## Part 1: Source Definition and Emission Calculation

In this section, we'll define radiation sources and calculate their emissions using the ENSDF decay data library.

In [ ]:
# Define sources and activities
print("Creating radiation sources...")

# Set activity units
Bq = ActivityUnit.Bq
uCi = ActivityUnit.uCi

# Create sources (Cs-137 with Ba-137m daughter in equilibrium)
Cs137 = SourceImpl.fromActivity(Nuclides.get("Cs137"), 1.0, uCi)  # 1 μCi of Cs-137
Ba137m = SourceImpl.fromActivity(Nuclides.get("Ba137m"), 0.947, uCi)  # 94.7% branching ratio to Ba-137m

# Add to source list
sourceList = ArrayList()
sourceList.add(Cs137)
sourceList.add(Ba137m)

print(f"Created Cs-137 source with {Cs137.getActivity().getValue()} {Cs137.getActivity().getUnits()}")
print(f"Created Ba-137m source with {Ba137m.getActivity().getValue()} {Ba137m.getActivity().getUnits()}")

In [ ]:
# Initialize the BNL decay library
print("Loading nuclear decay data...")
bnllib = BNLDecayLibrary()
bnllib.loadFile(Paths.get("BNL2023.txt"))  # Ensure this file is in your working directory

# Create emission calculator with the decay library
emcal = EmissionCalculator()
emcal.setDecayLibrary(bnllib)

# Calculate emissions from the source list
print("Calculating emissions...")
emissions = emcal.calculate(sourceList)

In [ ]:
# Process and display emission data
gamma_data = {'Energy': [], 'Intensity': []}
beta_data = {'Energy': [], 'Intensity': [], 'Forbiddenness': []}
xray_data = {'Energy': [], 'Intensity': [], 'Name': []}

# Extract gamma emissions
print("\nGamma Emissions:")
print("-" * 40)
print(f"{'Energy (keV)':<15} {'Intensity (/s)':<15}")
print("-" * 40)
for emission in emissions.getGammas():
    energy = emission.getEnergy().getValue()
    intensity = emission.getIntensity().getValue()
    gamma_data['Energy'].append(energy)
    gamma_data['Intensity'].append(intensity)
    print(f"{energy:<15.3f} {intensity:<15.3e}")

# Extract beta emissions
print("\nBeta Emissions:")
print("-" * 60)
print(f"{'Energy (keV)':<15} {'Intensity (/s)':<15} {'Forbiddenness':<15}")
print("-" * 60)
for emission in emissions.getBetas():
    energy = emission.getEnergy().getValue()
    intensity = emission.getIntensity().getValue()
    forbiddenness = emission.getForbiddenness() if emission.getForbiddenness() is not None else ""
    beta_data['Energy'].append(energy)
    beta_data['Intensity'].append(intensity)
    beta_data['Forbiddenness'].append(forbiddenness)
    print(f"{energy:<15.3f} {intensity:<15.3e} {forbiddenness:<15}")

# Extract X-ray emissions
print("\nX-ray Emissions:")
print("-" * 60)
print(f"{'Energy (keV)':<15} {'Intensity (/s)':<15} {'Transition':<15}")
print("-" * 60)
for emission in emissions.getXrays():
    energy = emission.getEnergy().getValue()
    intensity = emission.getIntensity().getValue()
    name = emission.getName()
    xray_data['Energy'].append(energy)
    xray_data['Intensity'].append(intensity)
    xray_data['Name'].append(name)
    if intensity > 0.01:  # Only show significant X-rays
        print(f"{energy:<15.3f} {intensity:<15.3e} {name:<15}")

In [ ]:
# Visualize emission spectrum
plt.figure(figsize=(12, 8))

# Plot gamma lines
plt.vlines(gamma_data['Energy'], [0] * len(gamma_data['Energy']), gamma_data['Intensity'], 
           linewidth=2, color='red', label='Gamma')

# Plot X-ray lines
plt.vlines(xray_data['Energy'], [0] * len(xray_data['Energy']), xray_data['Intensity'], 
           linewidth=2, color='blue', label='X-ray', alpha=0.7)

# Only show beta endpoint energies
plt.vlines(beta_data['Energy'], [0] * len(beta_data['Energy']), beta_data['Intensity'], 
           linewidth=2, color='green', linestyle='--', label='Beta Endpoint')

plt.xlabel('Energy (keV)')
plt.ylabel('Intensity (particles/s)')
plt.title('Cs-137/Ba-137m Emission Spectrum')
plt.yscale('log')
plt.xlim(-50, 1200)  # Adjust limits based on energy range
plt.ylim(1e-3, 5e4)  # Adjust based on intensity range
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Part 2: Transport Simulation with GEANT4

Now we'll set up a GEANT4 simulation to model radiation transport through materials. We'll define geometries, materials, and track particles.

In [ ]:
# Helper function to convert gamma list to flux format
def gamma_list_to_flux(gamma_list, resolution=1.0):
    """Convert a list of discrete gamma emissions to a continuous flux representation"""
    # Determine energy range based on gamma energies
    max_energy = 2000  # keV
    nBins = int(max_energy / resolution)
    
    # Create energy bins
    energies = [i * resolution for i in range(nBins + 1)]
    densities = [0.0] * len(energies)
    
    # Assign gammas to bins
    for gamma in gamma_list:
        energy = gamma.getEnergy().getValue()
        intensity = gamma.getIntensity().getValue()
        
        # Find the bin this gamma falls into
        lower_index = int(energy // resolution)
        if lower_index >= len(energies) - 1:
            continue
            
        upper_index = lower_index + 1
        
        # Distribute intensity proportionally between bins
        lower_fraction = (upper_index * resolution - energy) / resolution
        upper_fraction = (energy - lower_index * resolution) / resolution
        
        densities[lower_index] += intensity * lower_fraction
        densities[upper_index] += intensity * upper_fraction
    
    # Create flux representation
    flux = FluxTrapezoid()
    for i in range(1, len(energies)):
        flux.addPhotonGroup(FluxGroupTrapezoid(energies[i-1], energies[i], densities[i-1], densities[i]))
    
    return flux

In [ ]:
# Initialize GEANT4 transport simulation
print("Setting up GEANT4 transport simulation...")
generator = SourceGenerator()

# Convert gamma emissions to flux format
gamma_flux = gamma_list_to_flux(emissions.getGammas(), resolution=1.0)
generator.setFlux(gamma_flux)
generator.setSourceParticle("gamma")

# Set number of particles to simulate
gamma_sum = sum(gamma.getIntensity().getValue() for gamma in emissions.getGammas())
generator.setNumberOfParticle(10000)  # Simulate 10,000 particles

# Set source geometry (point source)
generator.setSphericalSource(True)
generator.setSourceRadius(0.1)  # 1mm radius source

# Initialize environment
generator.setEnvironment()

In [ ]:
# Define detector geometry: NaI(Tl) crystal
print("Configuring detector geometry...")

# Define NaI crystal elements
detector_elements = ArrayList()
detector_elements.add("Na")
detector_elements.add("I")

# Element stoichiometry
detector_multipliers = ArrayList()
detector_multipliers.add(1)  # 1 Na atom
detector_multipliers.add(1)  # 1 I atom

# NaI density
detector_density = 3.67  # g/cm³

# Cylindrical NaI detector (2" × 2")
detector_geometries = ArrayList()
detector_geometries.add(0.0)    # inner radius (solid)
detector_geometries.add(2.54)   # outer radius (1" = 2.54 cm)
detector_geometries.add(2.54)   # half-height (1")
detector_geometries.add(0.0)    # starting phi angle
detector_geometries.add(360.0)  # delta phi (full cylinder)
detector_geometries.add(0.0)    # rotation X
detector_geometries.add(0.0)    # rotation Y
detector_geometries.add(0.0)    # rotation Z
detector_geometries.add(0.0)    # position X
detector_geometries.add(0.0)    # position Y
detector_geometries.add(10.0)   # position Z (10 cm from source)

# Add detector to environment
generator.environment.addCylindricalObject(
    detector_elements,
    detector_multipliers,
    detector_density,
    detector_geometries
)

In [ ]:
# Add shielding: Lead block between source and detector
print("Adding lead shielding...")

# Lead element
shield_elements = ArrayList()
shield_elements.add("Pb")

# Element stoichiometry
shield_multipliers = ArrayList()
shield_multipliers.add(1)  # Pure Pb

# Lead density
shield_density = 11.35  # g/cm³

# Cuboid shield (1cm thick)
shield_geometries = ArrayList()
shield_geometries.add(5.0)    # half-width in x
shield_geometries.add(5.0)    # half-height in y
shield_geometries.add(0.5)    # half-thickness in z
shield_geometries.add(0.0)    # rotation X
shield_geometries.add(0.0)    # rotation Y
shield_geometries.add(0.0)    # rotation Z
shield_geometries.add(0.0)    # position X
shield_geometries.add(0.0)    # position Y
shield_geometries.add(5.0)    # position Z (5 cm from source)

# Add shield to environment using box object
generator.environment.addBoxObject(
    shield_elements,
    shield_multipliers,
    shield_density,
    shield_geometries
)

In [ ]:
# Configure simulation settings
print("Configuring and running GEANT4 simulation...")
generator.environment.prepareBeamLines()
generator.environment.writeSettingsToMacro()

# Execute GEANT4 simulation (first run checks geometry)
print("Checking geometry...")
runScript_path = './runGEANT4.sh'
try:
    result = subprocess.run([runScript_path], check=True, shell=True, text=True, capture_output=True)
    print("Geometry check complete.")
except subprocess.CalledProcessError as e:
    print(f"Geometry check failed with error: {e}")
    print("Error output:\n", e.stderr)

# Run full simulation with particles
print("Running full transport simulation...")
runScript_path = './runGEANT4.sh gamma'
try:
    result = subprocess.run([runScript_path], check=True, shell=True, text=True, capture_output=True)
    print("Transport simulation complete.")
except subprocess.CalledProcessError as e:
    print(f"Simulation failed with error: {e}")
    print("Error output:\n", e.stderr)

In [ ]:
# Parse simulation results
print("Processing simulation results...")
generator.environment.parseResults()

# Get energy deposition spectra
gamma_spectrum = generator.environment.getGammaSpectrum()
electron_spectrum = generator.environment.getElectronSpectrum()

# Helper function to extract spectrum data
def extract_spectrum_data(spectrum):
    energies = []
    counts = []
    for group in spectrum.getPhotonGroups():
        energies.append(group.getEnergyLower())
        counts.append(group.getCounts())
    # Add final upper energy
    if spectrum.getPhotonGroups().size() > 0:
        energies.append(spectrum.getPhotonGroups().get(spectrum.getPhotonGroups().size() - 1).getEnergyUpper())
    return energies, counts

# Extract data for plotting
gamma_energies, gamma_counts = extract_spectrum_data(gamma_spectrum)
electron_energies, electron_counts = extract_spectrum_data(electron_spectrum)

# Convert to incident flux for next stage
incident_flux = gamma_spectrum

In [ ]:
# Plot transport simulation results
plt.figure(figsize=(12, 6))

# Plot gamma spectrum as histogram
plt.step(gamma_energies[:-1], gamma_counts, where='post', color='red', label='Gamma')

# Plot electron spectrum
plt.step(electron_energies[:-1], electron_counts, where='post', color='blue', label='Electron')

plt.xlabel('Energy (keV)')
plt.ylabel('Counts')
plt.title('GEANT4 Transport Simulation Results')
plt.grid(True, alpha=0.3)
plt.legend()

# Use log scale if there are counts
if any(c > 0 for c in gamma_counts + electron_counts):
    plt.yscale('log')

plt.xlim(0, 1000)  # Focus on energy range of interest
plt.show()

## Part 3: Detector Response Simulation

Now we'll use the incident flux from the transport calculation to simulate the detector response using RadSim's flow matching technique.

In [ ]:
# Set up detector geometry for response calculation
print("Configuring detector response model...")

# Create detector geometry (NaI cylinder)
geometry = CylinderChordGeometry()
geometry.setRadius(Quantity.of(2.54, "cm"))  # 1" radius
geometry.setHeight(Quantity.of(5.08, "cm"))  # 2" height

# Create cross-section library for NaI
materialCrossSection = XCOMPhotonCrossSectionLibrary.getInstance()
Na = materialCrossSection.get(Elements.get("Na"))
I = materialCrossSection.get(Elements.get("I"))

# Create electron shell distribution for Compton scattering
electrons = ArrayList()

# Sodium electron shells
Na_n_electrons = [2, 8, 1]  # K, L, M shells
Na_binding_energies = [1072.0, 63.5, 30.4]  # eV

for n, energy in zip(Na_n_electrons, Na_binding_energies):
    energy_eV = Quantity.of(energy, "eV")
    electronShell = ElectronShell(n, energy_eV)
    electrons.add(electronShell)

# Iodine electron shells
I_n_electrons = [2, 8, 18, 18, 7]  # K, L, M, N, O shells
I_binding_energies = [33169.0, 5188.0, 1072.0, 186.0, 50.0]  # eV

for n, energy in zip(I_n_electrons, I_binding_energies):
    energy_eV = Quantity.of(energy, "eV")
    electronShell = ElectronShell(n, energy_eV)
    electrons.add(electronShell)

# Create Klein-Nishina distribution for Compton scattering
dbknDistribution = DBKNDistribution(electrons)

In [ ]:
# Create the DepositionCalculator
print("Initializing ML-accelerated DepositionCalculator...")

# Material density (NaI)
density = 3.67  # g/cm³
max_energy = 1000.0  # keV, maximum energy to simulate

# This is where the ML-powered flow matching happens
# The calculator uses neural networks to rapidly compute chord distributions
calculator = DepositionCalculator(
    Na,  # Material cross-sections
    geometry,  # Detector geometry
    dbknDistribution,  # Scattering distribution
    density,  # Material density
    max_energy  # Maximum energy
)

In [ ]:
# Set detector position relative to source
print("Setting detector position...")
calculator.setPosition(0.0, 0.0, 10.0)  # 10cm from source on z-axis

# Create energy binning for response calculation
energy_bins = 1024
bin_width = max_energy / energy_bins  # keV per bin

# Arrays to store results
energies = np.linspace(0, max_energy, energy_bins + 1)
response = np.zeros(energy_bins)

# Process each gamma energy
print("Simulating detector response...")
for gamma in emissions.getGammas():
    energy = gamma.getEnergy().getValue()
    intensity = gamma.getIntensity().getValue()
    
    if energy <= max_energy and intensity > 0.01:  # Only process significant gammas
        print(f"Processing gamma: {energy:.1f} keV, intensity: {intensity:.1e} /s")
        
        # Compute deposition for this gamma energy
        result = calculator.compute(Quantity.of(energy, "keV"), intensity)
        
        # Extract energy deposition spectrum
        spectrum = result.getSpectrum()
        
        # Add to response
        for i in range(min(len(spectrum), energy_bins)):
            response[i] += spectrum[i]

In [ ]:
# Apply detector resolution effects
print("Applying detector resolution effects...")

# NaI resolution function: sigma(E) = sqrt(a + b*E)
# Parameters for typical NaI detector
a = 1.0  # keV^2
b = 0.03  # keV

# Apply Gaussian broadening
response_with_resolution = np.zeros_like(response)
for i, E in enumerate(energies[:-1]):
    if response[i] > 0:
        sigma = np.sqrt(a + b * E)  # Energy-dependent resolution
        
        # Create Gaussian centered at this energy
        gauss = np.exp(-0.5 * ((energies[:-1] - E) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))
        gauss /= gauss.sum()  # Normalize
        
        # Convolve with response
        response_with_resolution += response[i] * gauss

# Final detector response spectrum
final_response = response_with_resolution

In [ ]:
# Plot detector response spectrum
plt.figure(figsize=(12, 6))

# Ideal response (no resolution effects)
plt.step(energies[:-1], response, where='post', color='blue', alpha=0.5, label='Ideal Response')

# Response with resolution effects
plt.step(energies[:-1], final_response, where='post', color='red', linewidth=2, label='NaI Response with Resolution')

# Mark key gamma energies
for gamma in emissions.getGammas():
    energy = gamma.getEnergy().getValue()
    intensity = gamma.getIntensity().getValue()
    if energy <= max_energy and intensity > 0.1:
        plt.axvline(x=energy, color='green', linestyle='--', alpha=0.7)
        plt.text(energy, plt.ylim()[1]*0.9, f"{energy:.1f} keV", rotation=90, alpha=0.7)

plt.xlabel('Energy (keV)')
plt.ylabel('Counts')
plt.title('Simulated NaI Detector Response to Cs-137/Ba-137m')
plt.grid(True, alpha=0.3)
plt.legend()
plt.xlim(0, 800)  # Focus on relevant energy range
plt.yscale('log')
plt.show()

## Complete End-to-End Workflow Summary

We've completed the full end-to-end simulation workflow using the RadSim framework:

1. **Source Definition**:
   - Defined Cs-137 and Ba-137m sources with appropriate activities
   - Used the BNL decay library to calculate emissions (gamma, beta, X-rays)

2. **Transport Simulation**:
   - Defined geometry including NaI detector and lead shielding
   - Used GEANT4 integration to simulate radiation transport
   - Calculated incident flux on the detector

3. **Detector Response**:
   - Used ML-accelerated flow matching for chord distributions
   - Calculated energy deposition in the detector
   - Applied detector resolution effects

The final spectrum shows the expected features of a Cs-137/Ba-137m spectrum in a NaI detector, including:
- 661.7 keV photopeak from Ba-137m decay
- Compton continuum
- Compton edge at ~478 keV
- Backscatter peak at ~184 keV
- X-ray peaks at lower energies

This demonstrates RadSim's ability to produce realistic gamma-ray detector output using its modular, open-source framework.